# Generation checks

Explore the ensemble contract produced by deterministic RDKit ETKDG generation.

In [2]:
# reload notebook imports upon module updates
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

#from ensemblelab import Ensemble, generate

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'ensemblelab').is_dir():
    raise RuntimeError('Open this notebook from within the ensemblelab repository.')
sys.path.insert(0, str(project_root / 'src'))




## Generate a small aligned ensemble

Every generated conformer has matching RDKit and ASE atom ordering. Energies remain unset until optimization methods are applied.

In [4]:
from ensemblelab.generators import Ensemble, generate

ensemble = generate('CCO', n_confs=3)

assert isinstance(ensemble, Ensemble)
assert ensemble.smiles == 'CCO'
assert ensemble.molecule.GetNumConformers() == 3
assert len(ensemble.conformers) == 3
assert [conformer.energy for conformer in ensemble.conformers] == [None, None, None]
assert ensemble.metadata['optimization_status'] == 'unoptimized'
assert ensemble.metadata['energy_status'] == 'uncomputed'

for conformer in ensemble.conformers:
    assert conformer.atoms.get_chemical_symbols() == [
        atom.GetSymbol() for atom in ensemble.molecule.GetAtoms()
    ]
    assert len(conformer.atoms) == ensemble.molecule.GetNumAtoms()
    assert ensemble.rdkit_conformer(conformer.id).GetId() == conformer.id

print('Generation contract passed.')
print(ensemble.metadata)

ensemble.show()


Generation contract passed.
{'n_conformers': 3, 'optimization_status': 'unoptimized', 'energy_status': 'uncomputed', 'energy_unit': None, 'rdkit_version': '2026.03.1', 'history': [{'process': 'generation', 'method': 'rdkit.ETKDGv3', 'requested_smiles': 'CCO', 'canonical_smiles': 'CCO', 'n_requested': 3, 'n_generated': 3, 'random_seed': 42, 'rdkit_version': '2026.03.1'}]}
Ensemble
----------------------------
SMILES: CCO
Atoms: 9
Conformers: 3
Energy: uncomputed
Optimization: not run

Conformers
------------------------------------
+----+--------+--------------------+--------+-----------+-------+
| ID | Energy | Delta E (kcal/mol) | Method | Converged | Atoms |
+----+--------+--------------------+--------+-----------+-------+
| 0  | N/A    | N/A                | N/A    | N/A       | 9     |
| 1  | N/A    | N/A                | N/A    | N/A       | 9     |
| 2  | N/A    | N/A                | N/A    | N/A       | 9     |
+----+--------+--------------------+--------+-----------+-------+


## Object-oriented constructor

`Ensemble.from_smiles()` is the object-oriented entry point and delegates to the same generator.

In [5]:
object_ensemble = Ensemble.from_smiles('CCO', n_confs=2)
assert len(object_ensemble.conformers) == 2
print(object_ensemble.conformer_ids)


(0, 1)


## Input validation

Invalid SMILES and invalid conformer counts fail explicitly rather than producing a partial ensemble.

In [6]:
for invalid_smiles in ('', 'not a smiles'):
    try:
        generate(invalid_smiles)
    except ValueError:
        print(f'Rejected invalid SMILES: {invalid_smiles!r}')
    else:
        raise AssertionError('Invalid SMILES should raise ValueError')

for n_confs in (0, -1, True, 1.5):
    try:
        generate('CCO', n_confs=n_confs)
    except ValueError:
        print(f'Rejected n_confs={n_confs!r}')
    else:
        raise AssertionError('Invalid conformer count should raise ValueError')

Rejected invalid SMILES: ''
Rejected invalid SMILES: 'not a smiles'
Rejected n_confs=0
Rejected n_confs=-1
Rejected n_confs=True
Rejected n_confs=1.5


[00:06:51] SMILES Parse Error: syntax error while parsing: not
[00:06:51] SMILES Parse Error: check for mistakes around position 3:
[00:06:51] not
[00:06:51] ~~^
[00:06:51] SMILES Parse Error: Failed parsing SMILES 'not' for input: 'not'
